
# Module 1 — Ungraded Lab (LO1)
**Draft a Measurement Plan for a Churn‑Save Model**  
> **Solution Notebook** — completed examples for each PRACTICE CHALLENGE.


In [ ]:

# ================================================
# Setup & Synthetic Data (Provided Example)
# ================================================
import numpy as np, pandas as pd

rng = np.random.default_rng(7)

N = 20_000  # users entering the funnel
regions = rng.choice(["NA","EU","LATAM","APAC"], size=N, p=[0.35,0.25,0.25,0.15])
assigned_variation = rng.choice(["control","treatment"], size=N)  # user-level assignment

# Exposure: fraction of assigned users actually receive/save offer (e.g., contactable + model threshold)
exposure_prob = np.where(assigned_variation=="treatment", 0.72, 0.05)  # small BAU exposure in control
exposed = rng.random(N) < exposure_prob

# Baseline churn and treatment effect (heterogeneous by region)
base_churn = np.select(
    [regions=="NA", regions=="EU", regions=="LATAM", regions=="APAC"],
    [0.14, 0.12, 0.18, 0.16],
    default=0.15
)

# Model effect only if exposed + treatment; effect sizes vary slightly by region
effect = np.select(
    [regions=="NA", regions=="EU", regions=="LATAM", regions=="APAC"],
    [0.020, 0.015, 0.030, 0.018],  # absolute churn reduction
    default=0.02
)

treated_effect = (assigned_variation=="treatment") & exposed
churn_prob = base_churn - (treated_effect * effect)

# 28d retention outcome
retained_28d = rng.random(N) > churn_prob

# Costs & guardrails
# Incentive only if exposed and accepted (approx using retained as proxy for acceptance for this synthetic demo)
incentive_cost = np.where(exposed & retained_28d, rng.normal(1.50, 0.35, N).clip(0.5, 3.0), 0.0)
nps_delta = np.where(exposed, rng.normal(0.05, 0.08, N), rng.normal(0.00, 0.05, N))  # slight lift when exposed
support_tickets = rng.poisson(lam=np.where(exposed, 0.010, 0.012), size=N)

# Revenue assumptions
arpu_28d = np.where(retained_28d, rng.normal(12.0, 2.0, N).clip(6, 24), 0.0)  # revenue in outcome window

df = pd.DataFrame({
    "user_id": np.arange(N),
    "region": regions,
    "assigned_variation": assigned_variation,
    "exposed": exposed,
    "retained_28d": retained_28d,
    "incentive_cost": incentive_cost,
    "nps_delta": nps_delta,
    "support_tickets": support_tickets,
    "arpu_28d": arpu_28d
})

# Timestamps to demonstrate exposure/outcome windows
start_ts = pd.Timestamp("2025-01-01")
df["exposure_ts"] = start_ts + pd.to_timedelta(rng.integers(0, 10, N), unit="D")
df["outcome_ts"]  = df["exposure_ts"] + pd.to_timedelta(28, unit="D")

print("Synthetic dataset shape:", df.shape)
df.head()


In [ ]:

# ================================================
# Helper Functions (Provided Example)
# ================================================
import pandas as pd

def compute_incremental_retained_revenue(df, counterfactual="BAU"):
    """Compute incremental retained revenue (IRR) under a chosen counterfactual.
    counterfactual: 'BAU' (business-as-usual leakage in control) or 'Holdout'
    Returns a dict with components and IRR.
    """
    d = df.copy()

    # Revenue in 28d window
    rev_treat = d.loc[d.assigned_variation=="treatment", "arpu_28d"].sum()
    rev_ctrl  = d.loc[d.assigned_variation=="control",   "arpu_28d"].sum()

    # Cost in 28d window (e.g., incentives)
    cost_treat = d.loc[d.assigned_variation=="treatment", "incentive_cost"].sum()
    cost_ctrl  = d.loc[d.assigned_variation=="control",   "incentive_cost"].sum()

    if counterfactual == "BAU":
        # Allow for small exposure in control (already present in synthetic generation)
        delta_rev = (rev_treat - rev_ctrl)
        delta_cost = (cost_treat - cost_ctrl)
    elif counterfactual == "Holdout":
        # Assume zero treatment exposure in control; recalibrate by removing control exposure/costs
        # (Proxy: treat control incentive as 0 and remove its exposed revenue uplift)
        ctrl = d.loc[d.assigned_variation=="control"].copy()
        # crude adjustment: zero out control exposure costs
        adjusted_ctrl_rev = ctrl["arpu_28d"].sum()
        adjusted_ctrl_cost = 0.0
        delta_rev = (rev_treat - adjusted_ctrl_rev)
        delta_cost = (cost_treat - adjusted_ctrl_cost)
    else:
        raise ValueError("counterfactual must be 'BAU' or 'Holdout'")

    irr = delta_rev - delta_cost
    return {
        "revenue_treatment": float(rev_treat),
        "revenue_control": float(rev_ctrl),
        "cost_treatment": float(cost_treat),
        "cost_control": float(cost_ctrl),
        "delta_revenue": float(delta_rev),
        "delta_cost": float(delta_cost),
        "IRR": float(irr)
    }

def windowed_retention(df, window_days=28):
    """Return a copy with retained_window indicator recomputed for a given window."""
    d = df.copy()
    # In this synthetic data retained_28d is already simulated; emulate sensitivity by modest noise
    if window_days < 28:
        d["retained_window"] = d["retained_28d"] & (d["exposed"])  # stricter proxy
    elif window_days > 28:
        d["retained_window"] = d["retained_28d"] | (d["exposed"] & (d["arpu_28d"]>0))
    else:
        d["retained_window"] = d["retained_28d"]
    return d

def guardrail_report(df, incentive_budget_per_saved=1.80, allow_nps_drop=False):
    """Compute simple PASS/FAIL guardrails."""
    d = df.copy()
    # Saved users proxy: retained under treatment minus control rate
    treat = d[d.assigned_variation=="treatment"]
    ctrl  = d[d.assigned_variation=="control"]
    lift = treat.retained_28d.mean() - ctrl.retained_28d.mean()
    saved_users = lift * len(treat)

    avg_incentive_per_saved = (treat.incentive_cost.sum() - ctrl.incentive_cost.sum()) / max(saved_users, 1e-6)

    nps_diff = treat.nps_delta.mean() - ctrl.nps_delta.mean()
    tickets_diff = treat.support_tickets.mean() - ctrl.support_tickets.mean()

    checks = {
        "NPS non-negative": (nps_diff >= 0) if not allow_nps_drop else True,
        "Incentive within budget": (avg_incentive_per_saved <= incentive_budget_per_saved),
        "Support tickets not higher": (tickets_diff <= 0)
    }
    final = "GO" if all(checks.values()) else "NO-GO"
    return {
        "lift_retained_28d": float(lift),
        "saved_users_est": float(saved_users),
        "avg_incentive_per_saved": float(avg_incentive_per_saved),
        "nps_diff": float(nps_diff),
        "tickets_diff": float(tickets_diff),
        "checks": checks,
        "decision": final
    }
print("Helpers ready.")



## Activity 1 — Metric Tree (Model → Product → Business)
Complete the **metric_tree** to connect technical metrics to P&L outcomes.


In [ ]:

### PRACTICE CHALLENGE 1 ###
#### TASK: Build a metric_tree that links model metrics → product KPIs → business outcomes.
# ---------- SOLUTION CODE ----------
metric_tree = {
    "model_metrics": {
        "auc": "Higher AUC improves ranking quality, enabling better targeting of save actions",
        "precision": "Higher precision reduces wasted outreach/incentives on non-churners",
        "recall": "Higher recall increases coverage of true churners, raising potential saves"
    },
    "product_kpis": {
        "successful_saves": "Recall × contactability × acceptance × execution quality",
        "net_retained_users": "Successful saves minus any induced churn from bad outreach"
    },
    "business_outcomes": {
        "incremental_retained_revenue": "Net retained users × ARPU_28d (or gross margin)"
    },
    "costs": {
        "incentive_burn": "Sum of incentives for exposed/accepted users",
        "ops_load_minutes": "Agent minutes for outreach/handling, impacts capacity/cost"
    }
}
metric_tree



## Activity 2 — Counterfactual & Primary Metric (IRR)
Select a **counterfactual** (BAU or Holdout), compute **Incremental Retained Revenue (IRR)**, and justify your choice.


In [ ]:

### PRACTICE CHALLENGE 2 ###
#### TASK: Pick a counterfactual and compute IRR. Justify your choice in markdown in the next cell.
# ---------- SOLUTION CODE ----------
counterfactual_choice = "BAU"  # defensible here given small background exposure in control
irr_result = compute_incremental_retained_revenue(df, counterfactual=counterfactual_choice)
irr_result



**Justification (example):**  
- **Choice:** *BAU* — because a nonzero background save program already exists (small leakage in control).  
- **Risks:** Spillovers and misattribution if agents prioritize high‑risk users outside assignment.  
- **Mitigation:** Audit routing rules, monitor cross‑arm contact rates, consider a **geo holdout** in a follow‑up test.



## Activity 3 — Exposure & Outcome Windows
Use the helper to recompute retention at **21d**, **28d**, and **35d** and compare lift.


In [ ]:

### PRACTICE CHALLENGE 3 ###
#### TASK: Compute lift for 21, 28, and 35-day windows and assemble a small comparison table.
# ---------- SOLUTION CODE ----------
rows = []
for wd in [21, 28, 35]:
    d = windowed_retention(df, window_days=wd)
    treat = d[d.assigned_variation=="treatment"]
    ctrl  = d[d.assigned_variation=="control"]
    lift = treat.retained_window.mean() - ctrl.retained_window.mean()
    rows.append({"window_days": wd, "lift_retained": float(lift)})
pd.DataFrame(rows)



## Activity 4 — Guardrails & Decision Rules
Evaluate NPS, incentive burn, and support tickets. Produce a GO/NO‑GO.


In [ ]:

### PRACTICE CHALLENGE 4 ###
#### TASK: Run guardrail checks and print a compact report.
# ---------- SOLUTION CODE ----------
report = guardrail_report(df, incentive_budget_per_saved=1.80, allow_nps_drop=False)
report



## Activity 5 — Instrumentation Checklist
Assert that required fields exist; add at least one audit field and validate.


In [ ]:

### PRACTICE CHALLENGE 5 ###
#### TASK: Validate required columns and add an audit field.
# ---------- SOLUTION CODE ----------
required = [
    "user_id","assigned_variation","exposed","exposure_ts",
    "retained_28d","incentive_cost","nps_delta","support_tickets"
]
exists = {c: (c in df.columns) for c in required}
exists


In [ ]:

# Add an audit field and validate again
# ---------- SOLUTION CODE ----------
df["model_version"] = "churnsave_v1"
"model_version" in df.columns


> End of Solution.